## Course map

| Notebook | Main focus |
|---|---|
| Beginner | Installation, ONNX validation, JSON basics, first compile, Model Zoo |
| Intermediate | Calibration quality, hardware PPU, YOLO26 TopK optimization |
| Advanced | Q-PRO, diagnosis, QXNN resume, QAT, Python API, dvanced compiler controls |

Complete the notebooks in order unless you already understand DX-COM configuration and calibration.


# DX-COM Tutorial 3: Advanced

This tutorial covers accuracy recovery, quantization diagnosis, reusable QXNN checkpoints, Quantization-Aware Training (QAT), and programmatic compilation with the `dx_com` Python API.

The first labs use SqueezeNet so that you can focus on the workflow. The final lab uses a compact two-input stereo-fusion model to demonstrate a case that requires the Python API.

> **Before you begin:** Complete the Beginner and Intermediate tutorials first. Compilation can take several minutes. Every compile cell checks for an existing DXNN and skips repeated work.


## Learning objectives

By the end of this tutorial, you will be able to:

- compare Q-Lite PTQ, Q-PRO enhanced PTQ, and Q-Master QAT and select an appropriate workflow,
- build controlled Q-Lite and automatic Q-PRO experiments from the same model, data, and preprocessing,
- use a quantization diagnosis report to identify one-variable-at-a-time accuracy-recovery experiments,
- reuse a QXNN checkpoint for IQR recalibration and automatic Q-PRO without repeating the full ONNX compile,
- prepare a Q-Master/QAT configuration and distinguish a pipeline smoke test from meaningful accuracy training,
- choose between the `dxcom` CLI and `dx_com.compile()` Python API based on model and workflow requirements,
- implement a name-keyed PyTorch `DataLoader` and compile a two-input model with the Python API,
- apply advanced compiler controls only when they answer a specific graph or performance question, and
- collect model, calibration, compiler, accuracy, performance, and reproducibility evidence for release review.

This notebook does not modify the SDK source tree. All generated files are stored under `<dx-tutorials>/notebooks/T05-DX-Compiler`.

## 1. Initialize the tutorial workspace

The setup cell resolves every SDK path from `config.json`, creates an isolated workspace, and reuses the SDK calibration images through a symbolic link.

Generated tutorial files are stored under:

```text
<dx-tutorials>/notebooks/T05-DX-Compiler/
├── models/
├── configs/
├── outputs/
└── calibration_dataset -> <DX_COM_DIR>/calibration_dataset
```


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys

# Find the dx-tutorials root regardless of where JupyterLab was opened.
TUTORIAL_ROOT = Path.cwd().resolve()
while TUTORIAL_ROOT != TUTORIAL_ROOT.parent:
    if (TUTORIAL_ROOT / "tutorial_paths.py").is_file():
        break
    TUTORIAL_ROOT = TUTORIAL_ROOT.parent
else:
    raise FileNotFoundError(
        "tutorial_paths.py was not found. Start JupyterLab with ./run-jupyter-lab.sh."
    )

sys.path.insert(0, str(TUTORIAL_ROOT))
from tutorial_paths import DX_ALL_SUITE_DIR, DX_COMPILER_DIR

DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXCOM_PYTHON = DX_COMPILER_VENV / "bin" / "python"
DXPARSE_PATH = shutil.which("dxparse")

for required_path in (DX_COM_DIR, DXCOM_PATH, DXCOM_PYTHON):
    if not required_path.exists():
        raise FileNotFoundError(f"Required DX-COM path does not exist: {required_path}")
if not DXPARSE_PATH:
    raise FileNotFoundError("dxparse was not found in PATH. Install DX-RT first.")

WORK_DIR = TUTORIAL_ROOT / "notebooks/T05-DX-Compiler"
MODEL_DIR = WORK_DIR / "models"
CONFIG_DIR = WORK_DIR / "configs"
OUTPUT_DIR = WORK_DIR / "outputs"

for path in (WORK_DIR, MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

CALIBRATION_SOURCE = DX_COM_DIR / "calibration_dataset"
CALIBRATION_DIR = WORK_DIR / "calibration_dataset"
if not CALIBRATION_SOURCE.is_dir():
    raise FileNotFoundError(f"Calibration dataset was not found: {CALIBRATION_SOURCE}")
if not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.symlink_to(CALIBRATION_SOURCE, target_is_directory=True)

os.chdir(WORK_DIR)
print(f"DX-COM Python : {DXCOM_PYTHON}")
print(f"DX-COM CLI    : {DXCOM_PATH}")
print(f"Workspace     : {WORK_DIR}")
print(f"Calibration   : {CALIBRATION_DIR} -> {CALIBRATION_SOURCE}")


The Jupyter kernel and the DX-COM compiler use separate Python environments. This is intentional:

```text
Jupyter code       -> <dx-tutorials>/.venv/bin/python
DX-COM Python API  -> <DX_COMPILER_DIR>/venv-dx-compiler-local/bin/python
```

Install notebook-only inspection packages into the Jupyter environment with `uv`. Do not use `%pip` in this uv-managed environment.


In [ ]:
!uv pip install --python "{sys.executable}" --quiet onnx numpy

import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper

print("Notebook Python:", sys.executable)
print("ONNX version  :", onnx.__version__)


## 2. Prepare a compact reference model

We use the Model Zoo SqueezeNet ONNX and its matching Q-Lite JSON. The downloaded preprocessing remains unchanged; only the local calibration-dataset path is replaced.

The downloader follows two rules:

1. If a non-empty destination file exists, skip the download without a network request.
2. For a new file, download to `.part` and replace the destination only after the size check passes.

Delete a local file first when you intentionally need a fresh copy.


In [ ]:
from urllib.request import Request, urlopen


def download_file(url, destination, chunk_size=1024 * 1024):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.is_file() and destination.stat().st_size > 0:
        print(f"Skip download: found {destination} ({destination.stat().st_size:,} bytes)")
        return destination

    if destination.is_file():
        print(f"Existing file is empty; downloading again: {destination}")

    with urlopen(Request(url, method="HEAD"), timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length") or 0)

    temporary = destination.with_name(destination.name + ".part")
    temporary.unlink(missing_ok=True)
    downloaded = 0
    next_report = 10

    try:
        with urlopen(url, timeout=60) as response, temporary.open("wb") as output:
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if expected_size:
                    percent = downloaded * 100 // expected_size
                    if percent >= next_report:
                        print(f"{destination.name}: {min(percent, 100)}%")
                        next_report += 10

        if expected_size and downloaded != expected_size:
            raise IOError(
                f"Incomplete download: expected {expected_size} bytes, received {downloaded}"
            )
        temporary.replace(destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f"Saved: {destination} ({downloaded:,} bytes)")
    return destination


In [ ]:
ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/squeezenet-1.0_224x224.onnx"
JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/squeezenet-1.0_224x224.json"
MODEL_PATH = MODEL_DIR / "squeezenet-1.0_224x224.onnx"
DOWNLOADED_CONFIG = CONFIG_DIR / "squeezenet.modelzoo.json"

download_file(ONNX_URL, MODEL_PATH)
download_file(JSON_URL, DOWNLOADED_CONFIG)

model = onnx.load(MODEL_PATH)
onnx.checker.check_model(model)
print("Inputs :", [(value.name, [d.dim_value for d in value.type.tensor_type.shape.dim]) for value in model.graph.input])
print("Outputs:", [value.name for value in model.graph.output])
print("ONNX validation passed.")


In [ ]:
base_config = json.loads(DOWNLOADED_CONFIG.read_text())
base_config["default_loader"]["dataset_path"] = "./calibration_dataset"
BASE_CONFIG_PATH = CONFIG_DIR / "squeezenet.local.json"
BASE_CONFIG_PATH.write_text(json.dumps(base_config, indent=2) + "\n")

print("Local config:", BASE_CONFIG_PATH)
print("Input       :", base_config["inputs"])
print("Calibration :", base_config.get("calibration_method", "ema"))
print("Samples     :", base_config.get("calibration_num", 100))
print("Dataset     :", base_config["default_loader"]["dataset_path"])


## 3. Choose a quantization strategy

The three paths all produce an INT8 DXNN, but they use different methods and require different amounts of data and compute.

<img src="assets/q-lite-q-pro-q-master.png" alt="Comparison of Q-Lite, Q-PRO, and Q-Master" style="max-width: 600px;">

| Item | Q-Lite | Q-PRO | Q-Master |
|---|---|---|---|
| Quantization method | Standard PTQ | Enhanced PTQ | QAT (Quantization-Aware Training) |
| Main mechanism | Calibrate floating-point ranges | Apply automatic or manual DXQ enhancement stages | Fine-tune a quantized student model |
| Required data | Representative calibration samples | The same representative calibration samples | Training and validation samples; labels are needed when task loss is used |
| Training | No | No | Yes |
| Typical compute cost | Lowest | Higher than Q-Lite | Highest; normally uses a CUDA GPU |
| DX-COM entry point | Normal compile | `--use_q_pro` or a manual DXQ scheme | Add a `qmaster` block to the config |
| Recommended role | Controlled baseline | First PTQ accuracy-recovery experiment | Use when PTQ still misses the accuracy target |

In this tutorial, **Q-Master** refers to the DX-COM QAT workflow enabled by the JSON `qmaster` block. Do not select a method by its name alone:

<img src="assets/quantization-strategy-decision-flow.png" alt="Decision flow from Q-Lite to Q-PRO and Q-Master" style="max-width: 800px; width: 100%;">


### 3.1 Establish the Q-Lite baseline

A fair baseline uses the same ONNX, calibration data, preprocessing, compiler version, and validation protocol as every later experiment.

The next cell is equivalent to entering these commands in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/squeezenet-1.0_224x224.onnx \
      -c configs/squeezenet.local.json \
      -o outputs/squeezenet_q_lite_baseline \
      --gen_log \
      --export_html
```

The Notebook resolves the real paths automatically. If the output directory already contains a DXNN, it skips compilation.


In [ ]:
BASELINE_OUTPUT = OUTPUT_DIR / "squeezenet_q_lite_baseline"
baseline_dxnn_files = list(BASELINE_OUTPUT.glob("*.dxnn"))

if baseline_dxnn_files:
    print(f"Skip compilation: found {baseline_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{MODEL_PATH}" \
            -c "{BASE_CONFIG_PATH}" \
            -o "{BASELINE_OUTPUT}" \
            --gen_log \
            --export_html

baseline_dxnn_files = list(BASELINE_OUTPUT.glob("*.dxnn"))
if not baseline_dxnn_files:
    raise FileNotFoundError("Q-Lite compilation did not produce a DXNN file.")
BASELINE_DXNN = max(baseline_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("Q-Lite DXNN:", BASELINE_DXNN)


The HTML summary and `compiler.log` are part of the experiment evidence. They do not replace task-accuracy validation, but they make the build reviewable and reproducible.


In [ ]:
print("Generated Q-Lite artifacts:")
for artifact in sorted(BASELINE_OUTPUT.rglob("*")):
    if artifact.is_file() and artifact.suffix in {".dxnn", ".html", ".log"}:
        print("-", artifact.relative_to(WORK_DIR))

!dxparse -m "{BASELINE_DXNN}" -v


## 4. Run automatic Q-PRO

Q-PRO tries quantization-enhancement stages selected from the DXQ family. `--use_q_pro` is the easiest starting point because DX-COM chooses the combination automatically.

Important rules:

- Keep the Q-Lite and Q-PRO inputs identical.
- `--use_q_pro` and a manually selected `enhanced_scheme` are mutually exclusive.
- Higher compile cost is justified only when measured task accuracy improves.

Equivalent terminal commands:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/squeezenet-1.0_224x224.onnx \
      -c configs/squeezenet.local.json \
      -o outputs/squeezenet_q_pro \
      --use_q_pro \
      --opt_level 1 \
      --gen_log \
      --export_html
```


In [ ]:
QPRO_OUTPUT = OUTPUT_DIR / "squeezenet_q_pro"
qpro_dxnn_files = list(QPRO_OUTPUT.glob("*.dxnn"))

if qpro_dxnn_files:
    print(f"Skip compilation: found {qpro_dxnn_files[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{MODEL_PATH}" \
            -c "{BASE_CONFIG_PATH}" \
            -o "{QPRO_OUTPUT}" \
            --use_q_pro \
            --opt_level 1 \
            --gen_log \
            --export_html

qpro_dxnn_files = list(QPRO_OUTPUT.glob("*.dxnn"))
if not qpro_dxnn_files:
    raise FileNotFoundError("Q-PRO compilation did not produce a DXNN file.")
QPRO_DXNN = max(qpro_dxnn_files, key=lambda path: path.stat().st_mtime_ns)
print("Q-PRO DXNN:", QPRO_DXNN)


### 4.1 Compare the experiments correctly

Use the same held-out validation dataset for both models. Record more than one number:

| Evidence | Question answered |
|---|---|
| Task metric | Did Q-PRO recover useful accuracy? |
| Compile time | What development cost was added? |
| DXNN size | Did the deployment artifact change materially? |
| `dxrun` throughput | Did isolated runtime throughput change? |
| End-to-end latency and CPU | Did the complete application improve? |

Q-PRO is not automatically better when Q-Lite already meets the product target.


## 5. Diagnose quantization loss

Use diagnosis after you can reproduce an ONNX-versus-DXNN accuracy gap. `--quant_diagnosis` generates:

- `quant_diagnosis/diagnosis_report.html`: per-region quality evidence and retry guidance,
- `quant_diagnosis/<model>.qxnn`: a reusable checkpoint for re-quantization.

Equivalent terminal commands:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/squeezenet-1.0_224x224.onnx \
      -c configs/squeezenet.local.json \
      -o outputs/squeezenet_diagnosis \
      --quant_diagnosis \
      --gen_log \
      --export_html
```

Compilation is skipped only when the DXNN, diagnosis HTML, and QXNN checkpoint all exist.


In [ ]:
DIAG_OUTPUT = OUTPUT_DIR / "squeezenet_diagnosis"
diag_dxnn_files = list(DIAG_OUTPUT.glob("*.dxnn"))
diagnosis_reports = list(DIAG_OUTPUT.glob("quant_diagnosis/diagnosis_report.html"))
checkpoints = list(DIAG_OUTPUT.glob("quant_diagnosis/*.qxnn"))

if diag_dxnn_files and diagnosis_reports and checkpoints:
    print(f"Skip compilation: found {diag_dxnn_files[0]}")
    print(f"Diagnosis report: {diagnosis_reports[0]}")
    print(f"QXNN checkpoint: {checkpoints[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom -m "{MODEL_PATH}" \
            -c "{BASE_CONFIG_PATH}" \
            -o "{DIAG_OUTPUT}" \
            --quant_diagnosis \
            --gen_log \
            --export_html

    diag_dxnn_files = list(DIAG_OUTPUT.glob("*.dxnn"))
    diagnosis_reports = list(DIAG_OUTPUT.glob("quant_diagnosis/diagnosis_report.html"))
    checkpoints = list(DIAG_OUTPUT.glob("quant_diagnosis/*.qxnn"))

if not (diag_dxnn_files and diagnosis_reports and checkpoints):
    raise FileNotFoundError("Diagnosis did not produce all expected DXNN, HTML, and QXNN files.")

QXNN_CHECKPOINT = max(checkpoints, key=lambda path: path.stat().st_mtime_ns)
print("Diagnosis report:", diagnosis_reports[0])
print("QXNN checkpoint:", QXNN_CHECKPOINT)


### 5.1 Read the report as an experiment plan

1. Find the regions marked **Warning** or **Critical**.
2. Check where floating-point and quantized behavior first diverge.
3. Read the evidence and recommended recompile intents.
4. Select one change, such as a calibration method or Q-PRO.
5. Resume from the QXNN checkpoint.
6. Measure the same held-out task metric again.

Change one variable at a time. If the dataset, preprocessing, observer, and compiler options all change together, you cannot identify which change helped.


## 6. Re-quantize with QXNN resume

QXNN resume skips the earlier ONNX compilation work and reruns the quantization-dependent stages:

```text
Normal compile: ONNX -> optimize/partition -> quantize -> codegen -> DXNN
                                |
                                +-> diagnosis QXNN checkpoint
                                             |
Resume:                             new quantization setting -> DXNN
```

Rules:

- `--checkpoint` and `--model_path` are mutually exclusive.
- The original config is embedded in the checkpoint, so `-c` is not required.
- Use `--dataset_path` only to intentionally override the embedded location.


### 6.1 Resume with IQR recalibration

Equivalent terminal command:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom --checkpoint outputs/squeezenet_diagnosis/quant_diagnosis/<model>.qxnn \
      -o outputs/squeezenet_resume_iqr \
      --recalibration_method iqr \
      --dataset_path calibration_dataset
```


In [ ]:
RESUME_IQR_OUTPUT = OUTPUT_DIR / "squeezenet_resume_iqr"
resume_iqr_dxnn = list(RESUME_IQR_OUTPUT.glob("*.dxnn"))

if resume_iqr_dxnn:
    print(f"Skip compilation: found {resume_iqr_dxnn[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom --checkpoint "{QXNN_CHECKPOINT}" \
            -o "{RESUME_IQR_OUTPUT}" \
            --recalibration_method iqr \
            --dataset_path "{CALIBRATION_DIR}"

resume_iqr_dxnn = list(RESUME_IQR_OUTPUT.glob("*.dxnn"))
if not resume_iqr_dxnn:
    raise FileNotFoundError("IQR resume did not produce a DXNN file.")
print("IQR resume DXNN:", resume_iqr_dxnn[0])


### 6.2 Resume with automatic Q-PRO

This experiment reuses the same checkpoint but applies automatic Q-PRO instead of IQR.

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom --checkpoint outputs/squeezenet_diagnosis/quant_diagnosis/<model>.qxnn \
      -o outputs/squeezenet_resume_q_pro \
      --use_q_pro \
      --dataset_path calibration_dataset
```


In [ ]:
RESUME_QPRO_OUTPUT = OUTPUT_DIR / "squeezenet_resume_q_pro"
resume_qpro_dxnn = list(RESUME_QPRO_OUTPUT.glob("*.dxnn"))

if resume_qpro_dxnn:
    print(f"Skip compilation: found {resume_qpro_dxnn[0]}")
else:
    !source "{DX_COMPILER_VENV}/bin/activate" && \
      dxcom --checkpoint "{QXNN_CHECKPOINT}" \
            -o "{RESUME_QPRO_OUTPUT}" \
            --use_q_pro \
            --dataset_path "{CALIBRATION_DIR}"

resume_qpro_dxnn = list(RESUME_QPRO_OUTPUT.glob("*.dxnn"))
if not resume_qpro_dxnn:
    raise FileNotFoundError("Q-PRO resume did not produce a DXNN file.")
print("Q-PRO resume DXNN:", resume_qpro_dxnn[0])


## 7. Prepare a Q-Master (QAT) experiment

Q-Master uses Quantization-Aware Training when PTQ methods do not meet the accuracy target. Add a `qmaster` block to the normal JSON config; no separate CLI flag is required.

```text
Calibration -> QAT training -> best QXNN checkpoint -> final DXNN
```

A meaningful QAT run requires representative training and validation data and normally a CUDA GPU. The tutorial calibration images are not a training dataset, so this section creates a reviewed template instead of launching an expensive and misleading training run.

Key ideas:

- `default_loader` is the source of preprocessing for both calibration and training.
- `qmaster` contains training hyperparameters.
- `fast_run: true` validates the pipeline with one epoch and one batch; it does **not** validate accuracy.
- Preserve the dataset revision, config, training logs, best checkpoint, compiler version, and validation results.


In [ ]:
qat_config = json.loads(BASE_CONFIG_PATH.read_text())
qat_config["default_loader"]["dataset_path"] = "/path/to/representative/training/images"
qat_config["qmaster"] = {
    "batch_size": 16,
    "num_workers": 4,
    "train_limit": 500,
    "val_limit": 50,
    "device": "cuda:0",
    "epochs": 30,
    "lr": 1e-5,
    "optimizer": "adamw",
    "scheduler": "cosine",
    "use_amp": True,
    "use_kd": True,
    "kd_alpha": 1.0,
    "early_stopping_patience": 5,
}
QAT_CONFIG_PATH = CONFIG_DIR / "squeezenet.qat.template.json"
QAT_CONFIG_PATH.write_text(json.dumps(qat_config, indent=2) + "\n")

print("QAT template:", QAT_CONFIG_PATH)
print("Dataset     :", qat_config["default_loader"]["dataset_path"])
print("Q-Master   :", json.dumps(qat_config["qmaster"], indent=2))


After replacing the dataset path and reviewing the hyperparameters, run QAT in a separate terminal:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
dxcom -m models/squeezenet-1.0_224x224.onnx \
      -c configs/squeezenet.qat.template.json \
      -o outputs/squeezenet_qat \
      --gen_log \
      --export_html
```

Important outputs:

| Artifact | Purpose |
|---|---|
| `qat_checkpoint/qat_checkpoint.qxnn` | Best training checkpoint; reusable for compile-only resume |
| `*.dxnn` | Final deployment artifact |
| `compiler.log` and training log | Evidence for debugging and reproducibility |

For a pipeline smoke test only, add `"fast_run": true` inside `qmaster`. `fast_run` is a JSON key, not a `dxcom` option.


## 8. Compile programmatically with the Python API

The CLI and Python API use the same DX-COM compiler engine. The Python API does not make a standard model inherently faster or more accurate; it changes how your application supplies the model, calibration data, and compiler options.

### 8.1 When should you use each interface?

| Requirement | `dxcom` CLI | `dx_com.compile()` Python API |
|---|---:|---:|
| Standard single-image input with JSON preprocessing | Recommended | Supported |
| Reproducible shell command or CI step | Recommended | Supported |
| In-memory `onnx.ModelProto` | No | Yes |
| Custom PyTorch preprocessing/data pipeline | No | Yes |
| Non-image calibration data | No | Yes |
| Multi-input model | No | **Required** |
| Programmatic experiment loops and exception handling | Limited | Recommended |
| Direct QAT/checkpoint control from an application | Limited | Recommended |

Python API advantages:

- pass an ONNX path or an in-memory `onnx.ModelProto`,
- provide exact calibration tensors with a PyTorch `DataLoader`,
- support multi-input and non-image models,
- build repeatable experiment loops without shell parsing,
- handle Python exceptions and record structured metadata, and
- integrate compilation into model export, validation, and artifact-management code.

Two constraints prevent common mistakes:

1. Supply exactly one of `config=` or `dataloader=`; they are mutually exclusive.
2. With `dataloader=`, preprocessing must happen inside `Dataset.__getitem__()` and must match deployment preprocessing.


### 8.2 Inspect the installed API

The Notebook kernel does not import `dx_com` directly because DX-COM is installed in its own compiler environment. The next cell runs the equivalent of:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
python -c 'import inspect, dx_com; print(inspect.signature(dx_com.compile))'
```


In [ ]:
api_signature_command = [
    str(DXCOM_PYTHON),
    "-c",
    "import inspect, dx_com; print('dx_com version:', getattr(dx_com, '__version__', 'unknown')); print(inspect.signature(dx_com.compile))",
]
print("$", shlex.join(api_signature_command))
subprocess.run(api_signature_command, cwd=WORK_DIR, check=True)


The most important parameters are:

| Parameter | Meaning |
|---|---|
| `model` | ONNX file path or `onnx.ModelProto` |
| `output_dir` | Directory that receives DXNN and reports |
| `config` | JSON-driven calibration for a standard single-input image model |
| `dataloader` | Programmatic calibration tensors; required for this multi-input lab |
| `calibration_method` | `ema` or `minmax` for a normal compile |
| `calibration_num` | Number of samples consumed from the DataLoader |
| `opt_level` | `0` for a faster experiment or `1` for full optimization |
| `use_q_pro` | Enable automatic Q-PRO; cannot be combined with manual `enhanced_scheme` |
| `quant_diagnosis` | Generate diagnosis HTML and a resumable QXNN |
| `gen_log`, `export_html` | Preserve review and debugging evidence |

A standard single-input call looks like this:

```python
import dx_com

dx_com.compile(
    model="models/model.onnx",
    config="configs/model.json",
    output_dir="outputs/model",
    gen_log=True,
    export_html=True,
)
```

For a custom DataLoader, replace `config=` with `dataloader=`; never pass both.


### 8.3 A model that requires the Python API

Stereo depth, optical-flow, and feature-fusion networks commonly consume two synchronized tensors. For this lab, we select a compact stereo feature-fusion model because it demonstrates the multi-input requirement without a long model download or compile. The `dxcom` CLI's JSON loader supports one model input, so it cannot supply both tensors. The Python API can return a dictionary keyed by the exact ONNX input names:

```text
left image  -- preprocessing -- tensor "left_image"  --+
                                                        +--> stereo-fusion ONNX --> DXNN
right image -- preprocessing -- tensor "right_image" --+

Dataset.__getitem__()
    -> {"left_image": Tensor[C,H,W], "right_image": Tensor[C,H,W]}
DataLoader(batch_size=1)
    -> {"left_image": Tensor[1,C,H,W], "right_image": Tensor[1,C,H,W]}
```

The representative graph encodes both inputs before feature fusion. This avoids treating the two images as one input and preserves a genuine two-input DXNN contract:

```text
left_image  -> Conv -> left_features  --+
                                          Add -> ReLU -> fused_features
right_image -> Conv -> right_features --+
```

This lab builds a compact representative stereo-fusion graph. It is intentionally small so the exercise focuses on the API contract. Its synthetic calibration data validates compilation mechanics only; a production stereo model needs synchronized, representative left/right samples.


In [ ]:
STEREO_MODEL_PATH = MODEL_DIR / "stereo_fusion_16x16.onnx"

if STEREO_MODEL_PATH.is_file() and STEREO_MODEL_PATH.stat().st_size > 0:
    print(f"Reuse model: {STEREO_MODEL_PATH}")
else:
    left_input = helper.make_tensor_value_info(
        "left_image", TensorProto.FLOAT, [1, 3, 16, 16]
    )
    right_input = helper.make_tensor_value_info(
        "right_image", TensorProto.FLOAT, [1, 3, 16, 16]
    )
    output = helper.make_tensor_value_info(
        "fused_features", TensorProto.FLOAT, [1, 4, 16, 16]
    )

    rng = np.random.default_rng(2026)
    initializers = []
    for branch in ("left", "right"):
        initializers.extend([
            numpy_helper.from_array(
                rng.normal(0, 0.05, (4, 3, 3, 3)).astype(np.float32),
                name=f"{branch}_weight",
            ),
            numpy_helper.from_array(
                np.zeros(4, dtype=np.float32),
                name=f"{branch}_bias",
            ),
        ])

    nodes = [
        helper.make_node(
            "Conv",
            ["left_image", "left_weight", "left_bias"],
            ["left_features"],
            name="EncodeLeftImage",
            kernel_shape=[3, 3],
            pads=[1, 1, 1, 1],
        ),
        helper.make_node(
            "Conv",
            ["right_image", "right_weight", "right_bias"],
            ["right_features"],
            name="EncodeRightImage",
            kernel_shape=[3, 3],
            pads=[1, 1, 1, 1],
        ),
        helper.make_node(
            "Add",
            ["left_features", "right_features"],
            ["fused"],
            name="FuseStereoFeatures",
        ),
        helper.make_node(
            "Relu",
            ["fused"],
            ["fused_features"],
            name="ActivateFeatures",
        ),
    ]

    graph = helper.make_graph(
        nodes,
        "StereoFusionTutorial",
        [left_input, right_input],
        [output],
        initializers,
    )
    stereo_model = helper.make_model(
        graph,
        opset_imports=[helper.make_opsetid("", 18)],
        producer_name="dx-tutorials",
        ir_version=9,
    )
    onnx.checker.check_model(stereo_model)
    onnx.save(stereo_model, STEREO_MODEL_PATH)
    print("Saved model:", STEREO_MODEL_PATH)

stereo_model = onnx.load(STEREO_MODEL_PATH)
onnx.checker.check_model(stereo_model)
input_contract = {
    value.name: tuple(dim.dim_value for dim in value.type.tensor_type.shape.dim)
    for value in stereo_model.graph.input
}
expected_contract = {
    "left_image": (1, 3, 16, 16),
    "right_image": (1, 3, 16, 16),
}
if input_contract != expected_contract:
    raise ValueError(f"Unexpected stereo model inputs: {input_contract}")
print("Validated inputs:", input_contract)
print("Opset:", [(item.domain or "ai.onnx", item.version) for item in stereo_model.opset_import])


### 8.4 Implement the named calibration DataLoader

Each dataset item omits the batch dimension. `DataLoader(batch_size=1)` adds it automatically.

The dictionary keys are safer than a tuple:

- dictionary: inputs are matched by ONNX name,
- tuple/list: inputs are matched by the model's internal input order.

The script validates the first batch before calling the compiler, then skips compilation when a DXNN already exists.


In [ ]:
%%writefile compile_stereo_fusion.py
from pathlib import Path

import dx_com
import torch
from torch.utils.data import DataLoader, Dataset

WORK_DIR = Path(__file__).resolve().parent
MODEL_PATH = WORK_DIR / "models" / "stereo_fusion_16x16.onnx"
OUTPUT_DIR = WORK_DIR / "outputs" / "stereo_fusion_python_api"


class StereoCalibrationDataset(Dataset):
    """Deterministic tensors for API demonstration, not accuracy calibration."""

    def __len__(self):
        return 10

    def __getitem__(self, index):
        generator = torch.Generator().manual_seed(2026 + index)
        left = torch.rand((3, 16, 16), generator=generator, dtype=torch.float32)
        right = torch.clamp(
            left * 0.9
            + torch.rand((3, 16, 16), generator=generator, dtype=torch.float32) * 0.1,
            0.0,
            1.0,
        )
        return {
            "left_image": left,
            "right_image": right,
        }


def main():
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(f"Run Section 8.3 first: {MODEL_PATH}")

    dataset = StereoCalibrationDataset()
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    first_batch = next(iter(dataloader))
    actual_shapes = {name: tuple(tensor.shape) for name, tensor in first_batch.items()}
    expected_shapes = {
        "left_image": (1, 3, 16, 16),
        "right_image": (1, 3, 16, 16),
    }
    if actual_shapes != expected_shapes:
        raise ValueError(f"DataLoader does not match the ONNX inputs: {actual_shapes}")
    print("Validated DataLoader batch:", actual_shapes)

    existing_dxnn = list(OUTPUT_DIR.glob("*.dxnn"))
    if existing_dxnn:
        print(f"Skip compilation: found {existing_dxnn[0]}")
        return

    dx_com.compile(
        model=str(MODEL_PATH),
        output_dir=str(OUTPUT_DIR),
        dataloader=dataloader,
        calibration_method="ema",
        calibration_num=len(dataset),
        opt_level=0,
        gen_log=True,
        export_html=True,
    )

    generated_dxnn = list(OUTPUT_DIR.glob("*.dxnn"))
    if not generated_dxnn:
        raise FileNotFoundError("Python API compilation did not produce a DXNN file.")
    print("Generated DXNN:", generated_dxnn[0])


if __name__ == "__main__":
    main()


### 8.5 Compile with the DX-COM Python environment

The next Notebook cell runs the script with the compiler environment's Python interpreter. It is equivalent to:

```bash
source <DX_ALL_SUITE_DIR>/dx-compiler/venv-dx-compiler-local/bin/activate
cd <dx-tutorials>/T05-DX-Compiler
python compile_stereo_fusion.py
```

There is intentionally no `dxcom` CLI equivalent for this model: the compiler needs the two tensors supplied by the custom DataLoader.


In [ ]:
MULTI_INPUT_SCRIPT = WORK_DIR / "compile_stereo_fusion.py"
compile_api_command = [str(DXCOM_PYTHON), str(MULTI_INPUT_SCRIPT)]
print("$", shlex.join(compile_api_command))
subprocess.run(compile_api_command, cwd=WORK_DIR, check=True)

MULTI_INPUT_OUTPUT = OUTPUT_DIR / "stereo_fusion_python_api"
multi_input_dxnn_files = list(MULTI_INPUT_OUTPUT.glob("*.dxnn"))
if not multi_input_dxnn_files:
    raise FileNotFoundError("The Python API lab did not produce a DXNN file.")
MULTI_INPUT_DXNN = max(
    multi_input_dxnn_files,
    key=lambda path: path.stat().st_mtime_ns,
)
print("Multi-input DXNN:", MULTI_INPUT_DXNN)


### 8.6 Verify the result

`dxparse -v` should show two model inputs, `left_image` and `right_image`. This confirms that the compiled artifact preserves the multi-input interface.

The calibration DataLoader matches the ONNX contract (`float32`, NCHW). The compiled NPU interface may be reported as `uint8`, NHWC because DX-COM moves supported input conversion into the NPU path. For deployment, follow the input shape, dtype, and preprocessing guidance reported for the generated DXNN rather than assuming the ONNX layout is unchanged.


In [ ]:
!dxparse -m "{MULTI_INPUT_DXNN}" -v

print("Generated Python API artifacts:")
for artifact in sorted(MULTI_INPUT_OUTPUT.rglob("*")):
    if artifact.is_file() and artifact.suffix in {".dxnn", ".html", ".log"}:
        print("-", artifact.relative_to(WORK_DIR))


## 9. Use advanced compiler controls deliberately

These options should answer a specific graph, performance, or reproducibility question. Do not enable them as a generic “more optimization” bundle.

| CLI option | Python API parameter | Purpose | Main caution |
|---|---|---|---|
| `--opt_level {0,1}` | `opt_level` | Trade compile time for full optimization | Compare against a controlled baseline |
| `--aggressive_partitioning` | `aggressive_partitioning` | Explore more NPU partitioning | Experimental; validate outputs and end-to-end latency |
| `--compile_input_nodes` | `input_nodes` | Start at selected ONNX operator nodes | Changes the compiled interface |
| `--compile_output_nodes` | `output_nodes` | End at selected ONNX operator nodes | Remaining work moves to the host |
| `--float64_calibration` | `float64_calibration` | Improve cross-CPU calibration determinism | Higher compute and memory cost |
| `--gen_log` | `gen_log` | Preserve detailed compiler logs | Store with the released artifact |
| `--export_html` | `export_html` | Generate a self-contained summary | Does not replace task validation |

For subgraph compilation, use **ONNX operator node names**, not tensor names. Document the resulting input/output contract and the host-side operations that remain.


In [ ]:
!source "{DX_COMPILER_VENV}/bin/activate" && dxcom -h | \
  grep -E "aggressive_partitioning|compile_input_nodes|compile_output_nodes|float64_calibration|opt_level|export_html"


## 10. Build a release-validation record

A successful compiler exit is necessary, but it is not sufficient evidence for a release.

| Area | Minimum evidence |
|---|---|
| Source model | Export command, framework/version, ONNX checker result |
| Calibration | Dataset revision, sample count, preprocessing, method |
| Compiler | DX-COM version, full command/API arguments, logs, HTML report |
| Accuracy | Same task metric for ONNX and every DXNN candidate |
| Performance | Warm-up policy, duration, device, throughput, latency, CPU load |
| Integration | Input/output shapes, preprocessing ownership, decoder contract |
| Reproducibility | Artifact hashes and isolated output directories |

`dxrun` with dummy input is useful for throughput checks, but it cannot replace task accuracy or full application latency measurement.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("SqueezeNet ONNX :", sha256(MODEL_PATH))
print("SqueezeNet config:", sha256(BASE_CONFIG_PATH))
print("Stereo ONNX     :", sha256(STEREO_MODEL_PATH))


## 11. Summary

### 11.1. Advanced accuracy-recovery map

```text
Build a controlled Q-Lite baseline
                │
                ▼
       Accuracy target met?
          ┌─────┴─────┐
         Yes          No
          │            │
          │            ▼
          │      Try automatic Q-PRO
          │            │
          │            ▼
          │   Accuracy target met?
          │      ┌─────┴─────┐
          │     Yes          No
          │      │            │
          │      │            ▼
          │      │    Diagnose quantization loss
          │      │            │
          │      │            ▼
          │      │    Resume from QXNN and test
          │      │    one change at a time
          │      │            │
          │      │            ▼
          │      │    Prepare Q-Master QAT
          └──────┴────────────┘
                │
                ▼
 Validate accuracy, performance,
 integration, and reproducibility
```

### 11.2. Advanced workflow dashboard

| Workflow | When to use it | Main input | Main result |
|---|---|---|---|
| Q-Lite baseline | First controlled PTQ experiment | ONNX, JSON, representative calibration data | Baseline DXNN and evidence |
| Automatic Q-PRO | Q-Lite misses the accuracy target | Same model, data, preprocessing, and validation protocol | Enhanced PTQ candidate |
| Diagnosis and QXNN resume | The ONNX-to-DXNN accuracy gap is reproducible | Diagnosis report and QXNN checkpoint | Faster one-variable-at-a-time trials |
| Q-Master QAT | PTQ methods still miss the target | Representative training and validation data | Trained quantized candidate |
| Python API | Calibration requires custom tensors, non-image data, or multiple inputs | ONNX plus a name-keyed PyTorch DataLoader | Programmatically compiled DXNN |

### 11.3. Experiments completed

| Experiment | Controlled comparison or contract | Evidence produced |
|---|---|---|
| SqueezeNet Q-Lite vs Q-PRO | Same ONNX, calibration, preprocessing, and compiler settings | Separate DXNN files, logs, and HTML reports |
| Quantization diagnosis | Reproduce the accuracy-loss candidate before changing settings | Diagnosis HTML and reusable QXNN checkpoint |
| QXNN resume | IQR recalibration vs automatic Q-PRO from the same checkpoint | Faster quantization experiments |
| Q-Master preparation | Smoke-test configuration vs meaningful QAT training | Reviewed QAT template and validation requirements |
| Stereo-fusion Python API | Two named inputs supplied by one DataLoader | Two-input DXNN verified with `dxparse` |

### 11.4. Completion checklist

- [x] Built a reusable Q-Lite baseline
- [x] Ran automatic Q-PRO as a controlled PTQ experiment
- [x] Generated and interpreted a quantization diagnosis report
- [x] Reused a QXNN checkpoint for IQR and Q-PRO trials
- [x] Distinguished a Q-Master smoke test from meaningful QAT
- [x] Selected the CLI or Python API from workflow requirements
- [x] Implemented a name-keyed multi-input calibration DataLoader
- [x] Compiled and inspected a two-input DXNN
- [x] Identified the evidence required for release review
- [ ] Replace tutorial samples with representative product data
- [ ] Compare ONNX and every DXNN candidate with the same held-out task metric
- [ ] Measure full-application latency, throughput, and host CPU usage

### 11.5. How to choose

| If you need to... | Start with... |
|---|---|
| Establish a reproducible INT8 reference | Q-Lite |
| Recover PTQ accuracy without training | Automatic Q-PRO |
| Find where quantization loss begins | Quantization diagnosis |
| Test another quantization setting without repeating the full ONNX compile | QXNN resume |
| Recover accuracy after PTQ options are exhausted | Q-Master QAT |
| Supply custom, non-image, in-memory, or multiple named inputs | `dx_com.compile()` Python API |
| Build a reproducible shell or CI compile step for a standard model | `dxcom` CLI |

> **Remember:** a compiler success message, a generated DXNN, and a fast dummy-input benchmark are only partial evidence. Release decisions require representative data, the same held-out task metric for ONNX and DXNN, full-pipeline performance measurements, and reproducible artifacts.

### 11.6. Apply this workflow to your model

1. Freeze the source ONNX, preprocessing, dataset revision, and validation metric.
2. Establish the Q-Lite baseline before changing quantization settings.
3. Change one variable at a time and keep every result in an isolated output directory.
4. Escalate from Q-Lite to Q-PRO, diagnosis/QXNN resume, and finally Q-Master only when measured accuracy requires it.
5. Store commands or API arguments, compiler logs, reports, hashes, accuracy, and performance results with the release candidate.
